In [ ]:
from pyspark.sql.functions import col

base_path = "abfss://ecommerce@synapseeccomstorage.dfs.core.windows.net"

customers = spark.read.parquet(f"{base_path}/silver/customers")
orders = spark.read.parquet(f"{base_path}/silver/orders")
order_items = spark.read.parquet(f"{base_path}/silver/order_items")
products = spark.read.parquet(f"{base_path}/silver/products")
payments = spark.read.parquet(f"{base_path}/silver/order_payments")

dim_customer = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
).dropDuplicates()

dim_product = products.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
).dropDuplicates()

fact_sales = (
    orders.alias("o")
    .join(order_items.alias("oi"), col("o.order_id") == col("oi.order_id"), "inner")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("oi.product_id"),
        col("o.order_status"),
        col("o.order_purchase_timestamp"),
        col("oi.price"),
        col("oi.freight_value")
    )
    .withColumn("revenue", col("price") + col("freight_value"))
)

fact_payments = payments.select(
    "order_id",
    "payment_type",
    "payment_installments",
    "payment_value"
)

dim_customer.write.mode("overwrite").parquet(f"{base_path}/gold/dim_customer")
dim_product.write.mode("overwrite").parquet(f"{base_path}/gold/dim_product")
fact_sales.write.mode("overwrite").parquet(f"{base_path}/gold/fact_sales")
fact_payments.write.mode("overwrite").parquet(f"{base_path}/gold/fact_payments")

StatementMeta(, , -1, SessionError, , SessionError, True)

AVAILABLE_WORKSPACE_CAPACITY_EXCEEDED: Livy session has failed. Session state: Error. Error code: AVAILABLE_WORKSPACE_CAPACITY_EXCEEDED. Your job requested 12 vcores. However, the workspace only has 0 vcores available out of quota of 12 vcores for node size family [MemoryOptimized]. Try ending the running job(s), reducing the numbers of vcores requested or increasing your vcore quota. https://learn.microsoft.com/en-us/azure/synapse-analytics/spark/apache-spark-concepts#workspace-level Source: User.